# Hunt–Crossley Contact Model in OpenSim

## Overview and motivation

- Goal: model low-velocity, elastic vibroimpact between compact bodies where most strain energy is recovered but a small portion is dissipated.
- Empirical fact: coefficient of restitution e decreases with impact speed vi; energy loss scales ~ vi^3 for many materials in elastic range.
- Hertzian contact gives a nonlinear elastic force F = k x^(3/2) for spherical contacts; more general geometries can be modeled with F = k x^n, n ∈ [1, 1.5, 3/2], etc.

## Core Hunt–Crossley formulation

- Assumes a nonlinear elastic force law plus a state-dependent damping that vanishes at first touch and at peak compression.
Elastic law: F_elastic(x) = k x^n, where x is normal penetration (compression).
- Damping law: F_damp(x, ẋ) = λ x^n ẋ, i.e., viscous-like but scaled by the same x^n factor as the elastic term.
Governing ODE (free half-cycle of contact):
m ẍ + k x^n + λ x^n ẋ = 0
or equivalently m ẍ + k x^n (1 + (λ/k) ẋ) = 0.
- Special property: with λ chosen appropriately, the hysteresis loop area (energy loss ΔE) scales as ~ vi^3, matching experiments where e ≈ 1 − α vi for small vi.
λ is independent of exponent n for q=1 in Hunt–Crossley’s derivation; one obtains λ ≈ (3/2) α k, linking damping to the restitution slope α.


## Why it’s special

- Physically consistent onset: no tensile contact force; force starts at zero when x=0 and increases smoothly with compression; damping is zero at first touch and at maximum compression (ẋ=0).
- Energy-consistent scaling: reproduces ΔE ∝ vi^3 without invoking plasticity, reflecting micro-slip and internal dissipation mechanisms in elastic range.
- Geometry-flexible: compatible with different force–approach exponents (Hertzian n=3/2, cylinders n ∈ [1,1.5], flat contacts n≈1), avoiding over-reliance on a single geometry.
- Numerically stable for small penetrations when parameters are reasonable; avoids unphysical tensile forces and shock discontinuities typical of linear Kelvin–Voigt at impact onset.


## OpenSim HuntCrossleyForce

- OpenSim’s HuntCrossleyForce implements:
Nonlinear normal force: F_n = k x^n + λ x^n ẋ (applied only when penetration x>0).
- Dissipation parameter (“dissipation”) corresponds to λ/k scaling; OpenSim’s API exposes stiffness and dissipation as top-level parameters.
- Friction: static, dynamic, and viscous friction coefficients are included to model tangential contact (Coulomb-like plus viscous term). These act when bodies are in contact and depend on normal load.
- Key parameters in your code:
    - stiffness: N/m (or effective with exponent), sets k.
    - dissipation: non-dimensional damping scaling used to compute λ; higher values increase energy loss and reduce e at higher vi.
    - static_friction, dynamic_friction, viscous_friction: tangential friction behavior; set to zero if you want purely normal contact.

## Geometry setup:
- ContactSphere attached to the pendulum head;
- ContactHalfSpace as the wall;
- HuntCrossleyForce added with those geometries.
- Orientation/location of the half-space define when penetration occurs (x>0 for compression).

## Use cases
- Low-to-moderate speed impacts in elastic range: ball–plane, cylinder–plane, soft robotic bumpers, foot–ground contacts when plasticity is negligible.
- Systems where realistic restitution and energy loss scaling are desired without full viscoelastic material models.